In [ ]:
# import libraries
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import csv
import math
import numpy as np
import pandas as pd
import seaborn as sns
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, f1_score, average_precision_score
import pickle
import zipfile
import io
import os.path
import time
from multiprocessing import Pool

import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pack_padded_sequence
print("torch.__version__ =", torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device =", device)

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

import matplotlib
from matplotlib import pyplot
# %matplotlib inline
# from IPython import display
# display.set_matplotlib_formats('svg')

aa_list = ['_PAD',
           'A',
           'R',
           'N',
           'D',
           'C',
           'E',
           'Q',
           'G',
           'H',
           'I',
           'L',
           'K',
           'M',
           'F',
           'P',
           'S',
           'T',
           'W',
           'Y',
           'V',
          ]
vocab_size = len(aa_list)
aa2index = {}
index2aa = {}
for index, aa in enumerate(aa_list):
    aa2index[aa] = index
    index2aa[index] = aa

IEDB_response_code = {'Positive': 1,
                      'Positive-High': 1,
                      'Positive-Intermediate': 1,
                      'Positive-Low': 1,
                      'Negative': 0,
                     }
MAX_LEN = 14


In [ ]:
# prepare training data

training_data = "iedb_20250911"
cache_path = f"data/tcell/{training_data}.train_valid_test.pkl"
print("cache_path =", cache_path)

if os.path.isfile(cache_path):
    print("cache file exists, load train_set, valid_set, test_set")
    with open(cache_path, 'rb') as file:
        train_set, valid_set, test_set, train_set_pos, train_set_neg, eval_peptides = pickle.load(file)
    print("  len(train_set) = ", len(train_set))
    print("  len(valid_set) = ", len(valid_set))
    print("  len(test_set) = ", len(test_set))
    print("len(train_set_pos) =", len(train_set_pos))
    print("len(train_set_neg) =", len(train_set_neg))
    print()

else:
    # read IEDB T cell and MHC tables, filter unknown peptides
    tcell_table_file = "data/tcell/iedb.assay_tcell.hla_1.host_human/tcell_table_export_1757563718.csv"
    mhc_table_file = "data/tcell/iedb.assay_mhc.hla_1.host_human/mhc_ligand_full.20250908.csv"
    epitope_col = ('Epitope', 'Name')
    epitope_type_col = ('Epitope', 'Object Type')
    assay_col = ('Assay', 'Qualitative Measurement')
    class_col = ('MHC Restriction', 'Class')
    allele_col = ('MHC Restriction', 'Name')
    host_col = ('Host', 'Name')
    organism_col = ('Epitope', 'Source Organism')

    print("tcell_table_file =", tcell_table_file)
    tcell_df = pd.read_csv(tcell_table_file, header=[0, 1], keep_default_na=False)
    print("len(tcell_df) =", len(tcell_df))
    # filter unknown peptides
    tcell_df = tcell_df[tcell_df[epitope_type_col] == 'Linear peptide']
    tcell_df['unknown_aa'] = tcell_df.apply(lambda row: any([aa not in aa_list for aa in row[epitope_col]]), axis=1)
    tcell_df = tcell_df[~tcell_df['unknown_aa']]
    print("len(tcell_df) after filter unknown peptides =", len(tcell_df))
    tcell_df['response'] = tcell_df.apply(lambda row: IEDB_response_code[row[assay_col]], axis=1)
    print("  positive = ", sum(tcell_df['response']))
    print("  negative = ", sum(tcell_df['response'] == 0))
    print()

    print("mhc_table_file =", mhc_table_file)
    mhc_df = pd.read_csv(mhc_table_file, header=[0, 1], keep_default_na=False)
    print("len(mhc_df) =", len(mhc_df))
    mhc_df = mhc_df[(mhc_df[epitope_type_col] == 'Linear peptide') &
                    (mhc_df[assay_col].str.contains('Positive')) &
                    (mhc_df[class_col] == 'I') &
                    (mhc_df[host_col].str.contains('Homo sapiens')) &
                    (mhc_df[organism_col].str.contains('Homo sapiens')) &
                    True
                   ]
    mhc_df['unknown_aa'] = mhc_df.apply(lambda row: any([aa not in aa_list for aa in row[epitope_col]]), axis=1)
    mhc_df = mhc_df[~mhc_df['unknown_aa']]
    print("len(mhc_df) after filter unknown peptides =", len(mhc_df))
    print()

    tcell_triple = tcell_df[[epitope_col, allele_col, ('response', '')]]
    tcell_triple = tcell_triple.set_axis(['peptide', 'allele', 'response'], axis=1)
    print("len(tcell_triple) =", len(tcell_triple)); print(tcell_triple.head(1))
    mhc_pairs = mhc_df[[epitope_col, allele_col]]
    mhc_pairs = mhc_pairs.set_axis(['peptide', 'allele'], axis=1)
    print("len(mhc_pairs) = ", len(mhc_pairs)); print(mhc_pairs.head(1))
    print("reduce tcell_triple to sets of unique peptides")
    training_pos = set(tcell_triple[tcell_triple['response'] == 1]['peptide'])
    training_neg = set(tcell_triple[tcell_triple['response'] == 0]['peptide'])
    print("  len(training_pos) = ", len(training_pos))
    print("  len(training_neg) = ", len(training_neg))
    print()

    # add random MHC peptides to training_neg if needed
    neg_pos_ratio = 100
    if len(training_neg) <= neg_pos_ratio*len(training_pos):
        mhc_peptides = sorted(set(mhc_pairs['peptide']))
        print("len(mhc_peptides) = ", len(mhc_peptides))
        print("neg_pos_ratio =", neg_pos_ratio)
        print("add random MHC peptides to training_neg")
        random.seed(99)
        sample_size = min(neg_pos_ratio*len(training_pos) - len(training_neg), len(mhc_peptides))
        mhc_peptides_random = random.sample(mhc_peptides, k=sample_size)
        training_neg = training_neg.union(mhc_peptides_random)
        print("  len(mhc_peptides_random) = ", len(mhc_peptides_random))
        print("  len(training_pos) = ", len(training_pos))
        print("  len(training_neg) = ", len(training_neg))
        print()

    # filter out peptides from evaluation datasets
    print("filter out peptides from evaluation datasets")
    eval_files = [
        "test.chowell_4digit.csv",
        "test.mel_MS_6.csv",
        "test.pdac_ely_nc.csv",
        "test.prime2.csv",
    ]
    eval_peptides = []
    for f in eval_files:
        eval_peptides += list(pd.read_csv("data/tcell/" + f, keep_default_na=False)['peptide'])
    print("eval_files =", eval_files)
    print("len(eval_peptides) =", len(eval_peptides))
    eval_peptides = set(eval_peptides)
    print("len(eval_peptides) unique =", len(eval_peptides))
    print("exclude eval_peptides from both sets")
    training_pos = training_pos - eval_peptides
    training_neg = training_neg - eval_peptides
    print("  len(training_pos) = ", len(training_pos))
    print("  len(training_neg) = ", len(training_neg))
    print()

    overlap_peptides = training_pos.intersection(training_neg)
    print("len(overlap_peptides) = ", len(overlap_peptides))
    print("exclude overlap_peptides from training_neg")
    training_neg = training_neg - overlap_peptides
    print("  len(training_pos) = ", len(training_pos))
    print("  len(training_neg) = ", len(training_neg))
    print()

    training_pair_list = [[x, 1] for x in sorted(training_pos)] + [[x, 0] for x in sorted(training_neg)]
    print("len(training_pair_list) = ", len(training_pair_list))
    print("filter length_8_14")
    training_pair_list = [[x, y] for x, y in training_pair_list if len(x) >= 8 and len(x) <= 14]
    print("len(training_pair_list) = ", len(training_pair_list))
    print("training_pair_list[0] = ", training_pair_list[0])
    print()

    # train/valid/test split
    print("train/valid/test split")
    print("len(training_pair_list) = ", len(training_pair_list))
    train_valid_set, test_set = train_test_split(training_pair_list, test_size=0.1, random_state=99)
    train_set, valid_set = train_test_split(train_valid_set, test_size=0.1, random_state=99)
    print("  len(train_set) = ", len(train_set))
    print("  len(valid_set) = ", len(valid_set))
    print("  len(test_set) = ", len(test_set))
    train_set_pos = [x for x in train_set if x[1] == 1]
    train_set_neg = [x for x in train_set if x[1] == 0]
    print("len(train_set_pos) =", len(train_set_pos))
    print("len(train_set_neg) =", len(train_set_neg))
    print()

    print("save train_set, valid_set, test_set to cache_path")
    with open(cache_path, 'wb') as file:
        pickle.dump((train_set, valid_set, test_set, train_set_pos, train_set_neg, eval_peptides), file)
    print()

# sanity check
print("sanity check")
train_set_peptides = set([x[0] for x in (train_set_pos + train_set_neg)])
print("len(train_set_peptides) =", len(train_set_peptides))
print("len(eval_peptides) =", len(eval_peptides))
print("len(train_set_peptides.intersection(eval_peptides)) =", len(train_set_peptides.intersection(eval_peptides)))
print()


In [ ]:
##########################################################
# MODELS
##########################################################

In [ ]:
# Transformer model definition: 2 layers, 8 heads, 32 hidden units

class PositionalEmbedding(nn.Module):
    def __init__(self, vocab_size, max_len, d_model):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.position_embedding = nn.Embedding(max_len, d_model)

    def forward(self, x):
        positions = torch.arange(x.shape[1], device=x.device).unsqueeze(0)
        return self.token_embedding(x) + self.position_embedding(positions)


class TransformerModel(nn.Module):
    def __init__(self, vocab_size, max_len, num_hidden=32, num_heads=8, num_layers=2):
        super().__init__()
        self.embedding = PositionalEmbedding(vocab_size, max_len, num_hidden)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=num_hidden, nhead=num_heads, dim_feedforward=num_hidden, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.out = nn.Linear(max_len * num_hidden, 1)

    def forward(self, x):
        # amino acid index 0 is '_PAD', mask those positions out of self-attention
        padding_mask = (x == 0)
        embeddings = self.embedding(x)
        encoded = self.encoder(embeddings, src_key_padding_mask=padding_mask)
        return self.out(encoded.flatten(1)).squeeze(-1)

    def l2_penalty(self):
        return self.out.weight.pow(2).sum()

transformer_num_params = sum(p.numel() for p in TransformerModel(vocab_size, MAX_LEN, num_hidden=32, num_heads=8, num_layers=2).parameters())
print("TransformerModel num_params =", transformer_num_params)

def pad_sequences(sequences, maxlen):
    padded = np.zeros((len(sequences), maxlen), dtype=np.int64)
    for i, seq in enumerate(sequences):
        length = min(len(seq), maxlen)
        padded[i, :length] = seq[:length]
    return padded


def prepare_tensor(training_set):
    x_peptide = [x for x, y in training_set]
    y_tensor = np.array([y for x, y in training_set], dtype=np.float32)
    x_tensor = [[aa2index[aa] for aa in peptide] for peptide in x_peptide]
    x_tensor = pad_sequences(x_tensor, MAX_LEN)
    return x_tensor, y_tensor


In [ ]:
# MLP model definition, sized to match the Transformer's parameter count

class MLPModel(nn.Module):
    def __init__(self, vocab_size, max_len, embed_dim=32, hidden_dim=31):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc1 = nn.Linear(max_len * embed_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embeddings = self.embedding(x)
        hidden = F.relu(self.fc1(embeddings.flatten(1)))
        return self.out(hidden).squeeze(-1)

    def l2_penalty(self):
        return self.out.weight.pow(2).sum()


transformer_num_params = sum(p.numel() for p in TransformerModel(vocab_size, MAX_LEN, num_hidden=32, num_heads=8, num_layers=2).parameters())
mlp_num_params = sum(p.numel() for p in MLPModel(vocab_size, MAX_LEN).parameters())
print("TransformerModel num_params =", transformer_num_params)
print("MLPModel num_params =", mlp_num_params)


In [ ]:
# Bi-directional LSTM model definition, sized to match the other models' parameter count

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, max_len, embed_dim=16, hidden_dim=34):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True, bidirectional=True)
        self.out = nn.Linear(2 * hidden_dim, 1)

    def forward(self, x, lengths):
        embeddings = self.embedding(x)
        # amino acid index 0 is '_PAD'; pack so the LSTM skips padded timesteps
        packed = pack_padded_sequence(embeddings, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        hidden = torch.cat([h_n[0], h_n[1]], dim=1)  # concat forward/backward final hidden states
        return self.out(hidden).squeeze(-1)

    def l2_penalty(self):
        return self.out.weight.pow(2).sum()


lstm_num_params = sum(p.numel() for p in LSTMModel(vocab_size, MAX_LEN).parameters())
print("TransformerModel num_params =", transformer_num_params)
print("MLPModel num_params =", mlp_num_params)
print("LSTMModel num_params =", lstm_num_params)


In [ ]:
# CNN model definition, sized to match the other models' parameter count

class CNNModel(nn.Module):
    def __init__(self, vocab_size, max_len, embed_dim=32, num_channels=52, kernel_sizes=(3, 3)):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        k1, k2 = kernel_sizes
        self.conv1 = nn.Conv1d(embed_dim, num_channels, k1, padding=k1 // 2)
        self.conv2 = nn.Conv1d(num_channels, num_channels, k2, padding=k2 // 2)
        self.out = nn.Linear(num_channels * max_len, 1)

    def forward(self, x):
        embeddings = self.embedding(x).transpose(1, 2)  # (batch, embed_dim, max_len)
        hidden = F.relu(self.conv1(embeddings))
        hidden = F.relu(self.conv2(hidden))
        return self.out(hidden.flatten(1)).squeeze(-1)

    def l2_penalty(self):
        return self.out.weight.pow(2).sum()


cnn_num_params = sum(p.numel() for p in CNNModel(vocab_size, MAX_LEN).parameters())
print("TransformerModel num_params =", transformer_num_params)
print("MLPModel num_params =", mlp_num_params)
print("LSTMModel num_params =", lstm_num_params)
print("CNNModel num_params =", cnn_num_params)


In [ ]:
##########################################################
# TRAINING ensemble models
##########################################################

In [ ]:
# train an ensemble of 10 Transformer models (per-epoch negative downsampling, per original ensemble approach)

model_number = 10
model_dir = 'models/iedb_20250911.transformer_negdownsample_ensemble/'
os.makedirs(model_dir, exist_ok=True)
model_paths = [model_dir + f'model_{m}.pt' for m in range(model_number)]
print("model_paths =", model_paths)
print()

x_train_pos, y_train_pos = prepare_tensor(train_set_pos)
x_train_neg, y_train_neg = prepare_tensor(train_set_neg)
x_train_pos_t = torch.from_numpy(x_train_pos).long()
y_train_pos_t = torch.from_numpy(y_train_pos).float()
x_train_neg_t = torch.from_numpy(x_train_neg).long()
y_train_neg_t = torch.from_numpy(y_train_neg).float()

x_valid, y_valid = prepare_tensor(valid_set)
x_valid_t = torch.from_numpy(x_valid).long().to(device)
y_valid_t = torch.from_numpy(y_valid).float().to(device)

loss_fn = nn.BCEWithLogitsLoss()
train_neg_pos_ratio = 1
num_epochs = 10

elapsed_time = time.time()
for m, model_path in enumerate(model_paths):
    print("path =", model_path)
    if os.path.isfile(model_path):
        print("  model file exists, skip training")
        continue

    torch.manual_seed(99 + m)
    model = TransformerModel(vocab_size, MAX_LEN, num_hidden=32, num_heads=8, num_layers=2).to(device)
    optimizer = torch.optim.Adam(model.parameters())

    best_val_loss = float('inf')
    for epoch in range(num_epochs):
        # resample a fresh subset of negatives (matching train_neg_pos_ratio) to pair with all positives this epoch
        neg_idx = torch.randperm(len(train_set_neg))[:len(train_set_pos) * train_neg_pos_ratio]
        x_epoch = torch.cat([x_train_pos_t, x_train_neg_t[neg_idx]], dim=0)
        y_epoch = torch.cat([y_train_pos_t, y_train_neg_t[neg_idx]], dim=0)
        dataset = TensorDataset(x_epoch, y_epoch)
        loader = DataLoader(dataset, batch_size=32, shuffle=True)

        model.train()
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = loss_fn(logits, y_batch) + 0.01 * model.l2_penalty()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(x_valid_t)
            val_loss = loss_fn(val_logits, y_valid_t).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_path)
    print(f"  done, best_val_loss = {best_val_loss:.4f}")
elapsed_time = time.time() - elapsed_time
print("elapsed_time =", elapsed_time)


In [ ]:
# train an ensemble of 10 MLP models (per-epoch negative downsampling, per original ensemble approach)

model_number = 10
model_dir = 'models/iedb_20250911.mlp_negdownsample_ensemble/'
os.makedirs(model_dir, exist_ok=True)
model_paths = [model_dir + f'model_{m}.pt' for m in range(model_number)]
print("model_paths =", model_paths)
print()

x_train_pos, y_train_pos = prepare_tensor(train_set_pos)
x_train_neg, y_train_neg = prepare_tensor(train_set_neg)
x_train_pos_t = torch.from_numpy(x_train_pos).long()
y_train_pos_t = torch.from_numpy(y_train_pos).float()
x_train_neg_t = torch.from_numpy(x_train_neg).long()
y_train_neg_t = torch.from_numpy(y_train_neg).float()

x_valid, y_valid = prepare_tensor(valid_set)
x_valid_t = torch.from_numpy(x_valid).long().to(device)
y_valid_t = torch.from_numpy(y_valid).float().to(device)

loss_fn = nn.BCEWithLogitsLoss()
train_neg_pos_ratio = 1
num_epochs = 10

elapsed_time = time.time()
for m, model_path in enumerate(model_paths):
    print("path =", model_path)
    if os.path.isfile(model_path):
        print("  model file exists, skip training")
        continue

    torch.manual_seed(99 + m)
    model = MLPModel(vocab_size, MAX_LEN, embed_dim=32, hidden_dim=31).to(device)
    optimizer = torch.optim.Adam(model.parameters())

    best_val_loss = float('inf')
    for epoch in range(num_epochs):
        # resample a fresh subset of negatives (matching train_neg_pos_ratio) to pair with all positives this epoch
        neg_idx = torch.randperm(len(train_set_neg))[:len(train_set_pos) * train_neg_pos_ratio]
        x_epoch = torch.cat([x_train_pos_t, x_train_neg_t[neg_idx]], dim=0)
        y_epoch = torch.cat([y_train_pos_t, y_train_neg_t[neg_idx]], dim=0)
        dataset = TensorDataset(x_epoch, y_epoch)
        loader = DataLoader(dataset, batch_size=32, shuffle=True)

        model.train()
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = loss_fn(logits, y_batch) + 0.01 * model.l2_penalty()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(x_valid_t)
            val_loss = loss_fn(val_logits, y_valid_t).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_path)
    print(f"  done, best_val_loss = {best_val_loss:.4f}")
elapsed_time = time.time() - elapsed_time
print("elapsed_time =", elapsed_time)


In [ ]:
# train an ensemble of 10 LSTM models (per-epoch negative downsampling, per original ensemble approach)

model_number = 10
model_dir = 'models/iedb_20250911.lstm_negdownsample_ensemble/'
os.makedirs(model_dir, exist_ok=True)
model_paths = [model_dir + f'model_{m}.pt' for m in range(model_number)]
print("model_paths =", model_paths)
print()

x_train_pos, y_train_pos = prepare_tensor(train_set_pos)
x_train_neg, y_train_neg = prepare_tensor(train_set_neg)
x_train_pos_t = torch.from_numpy(x_train_pos).long()
y_train_pos_t = torch.from_numpy(y_train_pos).float()
lengths_train_pos_t = torch.from_numpy((x_train_pos != 0).sum(axis=1)).long()
x_train_neg_t = torch.from_numpy(x_train_neg).long()
y_train_neg_t = torch.from_numpy(y_train_neg).float()
lengths_train_neg_t = torch.from_numpy((x_train_neg != 0).sum(axis=1)).long()

x_valid, y_valid = prepare_tensor(valid_set)
x_valid_t = torch.from_numpy(x_valid).long().to(device)
y_valid_t = torch.from_numpy(y_valid).float().to(device)
lengths_valid = torch.from_numpy((x_valid != 0).sum(axis=1)).long()

loss_fn = nn.BCEWithLogitsLoss()
train_neg_pos_ratio = 1
num_epochs = 10

elapsed_time = time.time()
for m, model_path in enumerate(model_paths):
    print("path =", model_path)
    if os.path.isfile(model_path):
        print("  model file exists, skip training")
        continue

    torch.manual_seed(99 + m)
    model = LSTMModel(vocab_size, MAX_LEN, embed_dim=16, hidden_dim=34).to(device)
    optimizer = torch.optim.Adam(model.parameters())

    best_val_loss = float('inf')
    for epoch in range(num_epochs):
        # resample a fresh subset of negatives (matching train_neg_pos_ratio) to pair with all positives this epoch
        neg_idx = torch.randperm(len(train_set_neg))[:len(train_set_pos) * train_neg_pos_ratio]
        x_epoch = torch.cat([x_train_pos_t, x_train_neg_t[neg_idx]], dim=0)
        y_epoch = torch.cat([y_train_pos_t, y_train_neg_t[neg_idx]], dim=0)
        lengths_epoch = torch.cat([lengths_train_pos_t, lengths_train_neg_t[neg_idx]], dim=0)
        dataset = TensorDataset(x_epoch, y_epoch, lengths_epoch)
        loader = DataLoader(dataset, batch_size=32, shuffle=True)

        model.train()
        for x_batch, y_batch, len_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(x_batch, len_batch)
            loss = loss_fn(logits, y_batch) + 0.01 * model.l2_penalty()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(x_valid_t, lengths_valid)
            val_loss = loss_fn(val_logits, y_valid_t).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_path)
    print(f"  done, best_val_loss = {best_val_loss:.4f}")
elapsed_time = time.time() - elapsed_time
print("elapsed_time =", elapsed_time)


In [ ]:
# train an ensemble of 10 CNN models (per-epoch negative downsampling, per original ensemble approach)

model_number = 10
model_dir = 'models/iedb_20250911.cnn_negdownsample_ensemble/'
os.makedirs(model_dir, exist_ok=True)
model_paths = [model_dir + f'model_{m}.pt' for m in range(model_number)]
print("model_paths =", model_paths)
print()

x_train_pos, y_train_pos = prepare_tensor(train_set_pos)
x_train_neg, y_train_neg = prepare_tensor(train_set_neg)
x_train_pos_t = torch.from_numpy(x_train_pos).long()
y_train_pos_t = torch.from_numpy(y_train_pos).float()
x_train_neg_t = torch.from_numpy(x_train_neg).long()
y_train_neg_t = torch.from_numpy(y_train_neg).float()

x_valid, y_valid = prepare_tensor(valid_set)
x_valid_t = torch.from_numpy(x_valid).long().to(device)
y_valid_t = torch.from_numpy(y_valid).float().to(device)

loss_fn = nn.BCEWithLogitsLoss()
train_neg_pos_ratio = 1
num_epochs = 10

elapsed_time = time.time()
for m, model_path in enumerate(model_paths):
    print("path =", model_path)
    if os.path.isfile(model_path):
        print("  model file exists, skip training")
        continue

    torch.manual_seed(99 + m)
    model = CNNModel(vocab_size, MAX_LEN, embed_dim=32, num_channels=52, kernel_sizes=(3, 3)).to(device)
    optimizer = torch.optim.Adam(model.parameters())

    best_val_loss = float('inf')
    for epoch in range(num_epochs):
        # resample a fresh subset of negatives (matching train_neg_pos_ratio) to pair with all positives this epoch
        neg_idx = torch.randperm(len(train_set_neg))[:len(train_set_pos) * train_neg_pos_ratio]
        x_epoch = torch.cat([x_train_pos_t, x_train_neg_t[neg_idx]], dim=0)
        y_epoch = torch.cat([y_train_pos_t, y_train_neg_t[neg_idx]], dim=0)
        dataset = TensorDataset(x_epoch, y_epoch)
        loader = DataLoader(dataset, batch_size=32, shuffle=True)

        model.train()
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = loss_fn(logits, y_batch) + 0.01 * model.l2_penalty()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(x_valid_t)
            val_loss = loss_fn(val_logits, y_valid_t).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_path)
    print(f"  done, best_val_loss = {best_val_loss:.4f}")
elapsed_time = time.time() - elapsed_time
print("elapsed_time =", elapsed_time)


In [ ]:
##########################################################
# EVALUATING ensemble models
##########################################################

In [ ]:
# evaluate negdownsample ensembles (average of the 10 models per architecture) on the internal valid_set,
# and save each architecture's consensus prediction to a CSV alongside the peptide and true response

model_number = 10
negdownsample_ensembles = {
    'mlp': {
        'dir': 'models/iedb_20250911.mlp_negdownsample_ensemble/',
        'build': lambda: MLPModel(vocab_size, MAX_LEN, embed_dim=32, hidden_dim=31),
        'is_lstm': False,
    },
    'cnn': {
        'dir': 'models/iedb_20250911.cnn_negdownsample_ensemble/',
        'build': lambda: CNNModel(vocab_size, MAX_LEN, embed_dim=32, num_channels=52, kernel_sizes=(3, 3)),
        'is_lstm': False,
    },
    'lstm': {
        'dir': 'models/iedb_20250911.lstm_negdownsample_ensemble/',
        'build': lambda: LSTMModel(vocab_size, MAX_LEN, embed_dim=16, hidden_dim=34),
        'is_lstm': True,
    },
    'transformer': {
        'dir': 'models/iedb_20250911.transformer_negdownsample_ensemble/',
        'build': lambda: TransformerModel(vocab_size, MAX_LEN, num_hidden=32, num_heads=8, num_layers=2),
        'is_lstm': False,
    },
}

for name, info in negdownsample_ensembles.items():
    models = []
    for m in range(model_number):
        model = info['build']().to(device)
        model.load_state_dict(torch.load(info['dir'] + f'model_{m}.pt', map_location=device))
        model.eval()
        models.append(model)
    info['models'] = models
    print(name, ": loaded", len(models), "models from", info['dir'])
print()

x_valid, y_valid = prepare_tensor(valid_set)
x_valid_t = torch.from_numpy(x_valid).long().to(device)
lengths_valid = torch.from_numpy((x_valid != 0).sum(axis=1)).long()
print("len(valid_set) =", len(valid_set))
print()

print("=== valid_set AUC per negdownsample ensemble (consensus of 10) ===")
predictions = {}
for name, info in negdownsample_ensembles.items():
    y_pred_list = []
    for model in info['models']:
        with torch.no_grad():
            logits = model(x_valid_t, lengths_valid) if info['is_lstm'] else model(x_valid_t)
            y_pred_list.append(torch.sigmoid(logits).cpu().numpy())
    y_pred = np.mean(y_pred_list, axis=0)
    predictions[name] = y_pred
    auc = roc_auc_score(y_valid, y_pred)
    print(f"{name}_negdownsample (consensus of {len(info['models'])}) valid_auc = {auc:.4f}")

# consensus of all four architectures (average of their consensus predictions)
predictions['consensus'] = np.mean([predictions[name] for name in ['mlp', 'cnn', 'lstm', 'transformer']], axis=0)
auc = roc_auc_score(y_valid, predictions['consensus'])
print(f"consensus of 4 models (mlp, cnn, lstm, transformer) valid_auc = {auc:.4f}")

# consensus_rank: average percentile rank (0-1) of the four models' predictions for each peptide;
# this puts all four architectures on the same scale regardless of how spread out their raw scores are
percentile_ranks = {name: pd.Series(predictions[name]).rank(pct=True).values for name in ['mlp', 'cnn', 'lstm', 'transformer']}
predictions['consensus_rank'] = np.mean([percentile_ranks[name] for name in ['mlp', 'cnn', 'lstm', 'transformer']], axis=0)
auc = roc_auc_score(y_valid, predictions['consensus_rank'])
print(f"consensus_rank of 4 models (mlp, cnn, lstm, transformer) valid_auc = {auc:.4f}")
print()

# save peptide, true response, each architecture's consensus prediction, and both 4-model consensus scores
valid_predictions_path = "data/tcell/valid_set.negdownsample_ensemble_predictions.csv"
peptides = [x for x, y in valid_set]
valid_predictions_df = pd.DataFrame({'peptide': peptides, 'response': y_valid.astype(int)})
for name in ['mlp', 'cnn', 'lstm', 'transformer', 'consensus', 'consensus_rank']:
    valid_predictions_df[name] = predictions[name]
valid_predictions_df.to_csv(valid_predictions_path, index=False)
print("saved ->", valid_predictions_path)
print("len(valid_predictions_df) =", len(valid_predictions_df))


In [ ]:
# evaluate negdownsample ensembles (average of the 10 models per architecture) on held-out clinical/experimental test sets,
# and add each architecture's prediction plus the two 4-model consensus scores as new columns in each CSV file

model_number = 10
negdownsample_ensembles = {
    'mlp': {
        'dir': 'models/iedb_20250911.mlp_negdownsample_ensemble/',
        'build': lambda: MLPModel(vocab_size, MAX_LEN, embed_dim=32, hidden_dim=31),
        'is_lstm': False,
    },
    'cnn': {
        'dir': 'models/iedb_20250911.cnn_negdownsample_ensemble/',
        'build': lambda: CNNModel(vocab_size, MAX_LEN, embed_dim=32, num_channels=52, kernel_sizes=(3, 3)),
        'is_lstm': False,
    },
    'lstm': {
        'dir': 'models/iedb_20250911.lstm_negdownsample_ensemble/',
        'build': lambda: LSTMModel(vocab_size, MAX_LEN, embed_dim=16, hidden_dim=34),
        'is_lstm': True,
    },
    'transformer': {
        'dir': 'models/iedb_20250911.transformer_negdownsample_ensemble/',
        'build': lambda: TransformerModel(vocab_size, MAX_LEN, num_hidden=32, num_heads=8, num_layers=2),
        'is_lstm': False,
    },
}

for name, info in negdownsample_ensembles.items():
    models = []
    for m in range(model_number):
        model = info['build']().to(device)
        model.load_state_dict(torch.load(info['dir'] + f'model_{m}.pt', map_location=device))
        model.eval()
        models.append(model)
    info['models'] = models
    print(name, ": loaded", len(models), "models from", info['dir'])
print()

eval_test_files = [
    "test.mel_MS_6.csv", # canonical_ms dataset in the paper
    "test.pdac_ely_nc.csv", # noncanonical_ms dataset
    "test.prime2.csv", # predicted_mutHLAp dataset
    "test.chowell_4digit.csv", # pathogen_ms dataset
]

for f in eval_test_files:
    file_path = "data/tcell/" + f
    df = pd.read_csv(file_path, keep_default_na=False)
    print(f, ": n =", len(df))

    length_mask = df['peptide'].str.len() >= 8
    df_eval = df[length_mask]
    print("  n after filter length >= 8 =", len(df_eval))
    # peptides longer than MAX_LEN get truncated (kept the first MAX_LEN residues) by prepare_tensor/pad_sequences
    print("  n longer than MAX_LEN (truncated) =", int((df_eval['peptide'].str.len() > MAX_LEN).sum()))

    labels = df_eval['response'].str.lower().map({'positive': 1, 'negative': 0})
    print("  n positive =", int((labels == 1).sum()))
    print("  n negative =", int((labels == 0).sum()))

    eval_set = list(zip(df_eval['peptide'], labels))
    x_eval, y_eval = prepare_tensor(eval_set)
    x_eval_t = torch.from_numpy(x_eval).long().to(device)
    lengths_eval = torch.from_numpy((x_eval != 0).sum(axis=1)).long()

    eval_predictions = {}
    for name, info in negdownsample_ensembles.items():
        y_pred_list = []
        for model in info['models']:
            with torch.no_grad():
                logits = model(x_eval_t, lengths_eval) if info['is_lstm'] else model(x_eval_t)
                y_pred_list.append(torch.sigmoid(logits).cpu().numpy())
        y_pred = np.mean(y_pred_list, axis=0)
        eval_predictions[name] = y_pred
        auc = roc_auc_score(y_eval, y_pred)
        print(f"  {name}_negdownsample (consensus of {len(info['models'])}) AUC = {auc:.4f}")

    # consensus of all four architectures (average of their consensus predictions)
    eval_predictions['consensus'] = np.mean([eval_predictions[name] for name in ['mlp', 'cnn', 'lstm', 'transformer']], axis=0)
    auc = roc_auc_score(y_eval, eval_predictions['consensus'])
    print(f"  consensus of 4 models (mlp, cnn, lstm, transformer) AUC = {auc:.4f}")

    # consensus_rank: average percentile rank (0-1) of the four models' predictions for each peptide
    percentile_ranks = {name: pd.Series(eval_predictions[name]).rank(pct=True).values for name in ['mlp', 'cnn', 'lstm', 'transformer']}
    eval_predictions['consensus_rank'] = np.mean([percentile_ranks[name] for name in ['mlp', 'cnn', 'lstm', 'transformer']], axis=0)
    auc = roc_auc_score(y_eval, eval_predictions['consensus_rank'])
    print(f"  consensus_rank of 4 models (mlp, cnn, lstm, transformer) AUC = {auc:.4f}")

    # add the predictions as new columns, aligned back onto the original (unfiltered) dataframe by index;
    # rows filtered out above (peptide length < 8) get NaN for these columns
    for name in ['mlp', 'cnn', 'lstm', 'transformer', 'consensus', 'consensus_rank']:
        df.loc[df_eval.index, name] = eval_predictions[name]
    df.to_csv(file_path, index=False)
    print("  saved ->", file_path)
    print()


In [ ]:
##########################################################
# INTERPRETABILITY: position/amino-acid contribution via random-substitution ablation
##########################################################

In [ ]:
# derive a position/amino-acid contribution matrix for the MLP ensemble via random-amino-acid-substitution ablation:
# for each positive peptide in train_set, score it with the ensemble, then for each of the first 9 positions,
# replace that position's residue with a random *different* amino acid (averaged over 20 draws) and take the
# score difference; average the difference across all peptides sharing the same (amino acid, position)

NUM_POS = 9
NUM_DRAWS = 20

pos_peptides = [x for x, y in train_set_pos]
n_pos = len(pos_peptides)
print("num positive peptides in train_set =", n_pos)

x_train_pos, _ = prepare_tensor(train_set_pos)
x_t = torch.from_numpy(x_train_pos).long().to(device)

mlp_ensemble_dir = 'models/iedb_20250911.mlp_negdownsample_ensemble/'
mlp_models = []
for m in range(10):
    model = MLPModel(vocab_size, MAX_LEN, embed_dim=32, hidden_dim=31).to(device)
    model.load_state_dict(torch.load(mlp_ensemble_dir + f'model_{m}.pt', map_location=device))
    model.eval()
    mlp_models.append(model)

def ensemble_score(x_input, batch_size=4096):
    outs = []
    with torch.no_grad():
        for s in range(0, len(x_input), batch_size):
            batch = x_input[s:s + batch_size]
            preds = [torch.sigmoid(model(batch)).cpu().numpy() for model in mlp_models]
            outs.append(np.mean(preds, axis=0))
    return np.concatenate(outs)

baseline_scores = ensemble_score(x_t)

np.random.seed(99)
records = {}
for i in range(NUM_POS):
    orig_col = x_train_pos[:, i]
    rank = orig_col - 1  # 0..19 for a real residue, unused where the peptide is shorter than i+1

    diff_sum = np.zeros(n_pos)
    for d in range(NUM_DRAWS):
        rand = np.random.randint(0, 19, size=n_pos)
        shifted = np.where(rand >= rank, rand + 1, rand)  # 0..19, excludes 'rank'
        new_aa_idx = shifted + 1  # back to real-aa index range 1..20

        x_mod = x_t.clone()
        x_mod[:, i] = torch.from_numpy(new_aa_idx.astype(np.int64)).to(device)
        mod_scores = ensemble_score(x_mod)
        diff_sum += (baseline_scores - mod_scores)
    diff_avg = diff_sum / NUM_DRAWS

    for j, pep in enumerate(pos_peptides):
        if len(pep) > i:
            aa = pep[i]
            records.setdefault((aa, i + 1), []).append(diff_avg[j])
    print("  position", i + 1, "done")

rows = aa_list[1:]
cols = [f'pos_{i+1}' for i in range(NUM_POS)]
contribution_df = pd.DataFrame(index=rows, columns=cols, dtype=float)
count_df = pd.DataFrame(0, index=rows, columns=cols, dtype=int)
for (aa, pos), vals in records.items():
    contribution_df.loc[aa, f'pos_{pos}'] = np.mean(vals)
    count_df.loc[aa, f'pos_{pos}'] = len(vals)

print("min count =", count_df.values.min(), " max count =", count_df.values.max())

contribution_df['average'] = contribution_df.median(axis=1)
contribution_df.loc['average'] = contribution_df.median(axis=0)

out_path = 'data/tcell/mlp_negdownsample_ensemble.train_set_random_aa_contribution.csv'
contribution_df.to_csv(out_path)
print("saved ->", out_path)


In [ ]:
##########################################################
# COMBINING FEATURES: NetMHCpan score, dist_human, consensus
##########################################################

In [ ]:
# combine 'NetMHCpan score', 'consensus', 'dist_human', and 'dist_self' using per_file_features and two methods:
#   combined_lr:   5-fold CV logistic regression (standardized features)
#   combined_rank: average percentile rank (0-1) of the 3 features -- parameter-free, no CV needed

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

eval_test_files = [
    "test.mel_MS_6.csv", # canonical_ms dataset in the paper
    "test.pdac_ely_nc.csv", # noncanonical_ms dataset
    "test.prime2.csv", # predicted_mutHLAp dataset
    "test.chowell_4digit.csv", # pathogen_ms dataset
]

per_file_features = {
    "test.mel_MS_6.csv": ['consensus', 'dist_self'],
    "test.pdac_ely_nc.csv": ['consensus', 'dist_self'],
    "test.prime2.csv": ['NetMHCpan score', 'consensus'],
    "test.chowell_4digit.csv": ['consensus', 'dist_human'],
}
all_features = ['NetMHCpan score', 'consensus', 'dist_human', 'dist_self']

auc_table2 = pd.DataFrame(index=eval_test_files, columns=['n', 'n_pos'] + all_features + ['combined_lr', 'combined_rank'], dtype=float)
coef_table2 = pd.DataFrame(index=eval_test_files, columns=all_features, dtype=float)

for f in eval_test_files:
    file_path = "data/tcell/" + f
    df = pd.read_csv(file_path, keep_default_na=False)
    y = (df['response'].str.lower() == 'positive').astype(int).values
    features = per_file_features[f]

    auc_table2.loc[f, 'n'] = len(y)
    auc_table2.loc[f, 'n_pos'] = y.sum()
    for feat in features:
        auc_table2.loc[f, feat] = roc_auc_score(y, df[feat].values)

    # combined_lr: 5-fold CV logistic regression
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)
    X = df[features].values
    oof_pred = np.zeros(len(y))
    coefs = []
    for train_idx, test_idx in skf.split(X, y):
        scaler = StandardScaler().fit(X[train_idx])
        X_train = scaler.transform(X[train_idx])
        X_test = scaler.transform(X[test_idx])
        clf = LogisticRegression()
        clf.fit(X_train, y[train_idx])
        oof_pred[test_idx] = clf.predict_proba(X_test)[:, 1]
        coefs.append(clf.coef_[0])
    auc_table2.loc[f, 'combined_lr'] = roc_auc_score(y, oof_pred)
    coef_table2.loc[f, features] = np.mean(coefs, axis=0)

    # combined_rank: average percentile rank of the features used for this file
    percentile_ranks = [df[feat].rank(pct=True).values for feat in features]
    combined_rank = np.mean(percentile_ranks, axis=0)
    auc_table2.loc[f, 'combined_rank'] = roc_auc_score(y, combined_rank)

    df['combined_lr'] = oof_pred
    df['combined_rank'] = combined_rank
    df.to_csv(file_path, index=False)
    print("  saved ->", file_path)
    print()

print("=== AUC per feature, and combined, with per-file feature sets (NaN = feature not used for that file) ===")
print(auc_table2.T.round(4).to_string())
print()
print("=== avg standardized logistic regression coefficients (5-fold CV), per-file feature sets ===")
print(coef_table2.T.round(4).to_string())


In [ ]:
# NOT USED
print(stop)

# combine 'NetMHCpan score', 'consensus', and 'dist_human' using two methods:
#   combined_lr:   5-fold CV logistic regression (standardized features)
#   combined_rank: average percentile rank (0-1) of the 3 features -- parameter-free, no CV needed

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

combine_features = ['NetMHCpan score', 'consensus', 'dist_human']

eval_test_files = [
    "test.mel_MS_6.csv", # canonical_ms dataset in the paper
    "test.pdac_ely_nc.csv", # noncanonical_ms dataset
    "test.prime2.csv", # predicted_mutHLAp dataset
    "test.chowell_4digit.csv", # pathogen_ms dataset
]

auc_table = pd.DataFrame(index=eval_test_files, columns=['n', 'n_pos'] + combine_features + ['combined_lr', 'combined_rank'], dtype=float)
coef_table = pd.DataFrame(index=eval_test_files, columns=combine_features, dtype=float)

for f in eval_test_files:
    file_path = "data/tcell/" + f
    df = pd.read_csv(file_path, keep_default_na=False)
    y = (df['response'].str.lower() == 'positive').astype(int).values

    auc_table.loc[f, 'n'] = len(y)
    auc_table.loc[f, 'n_pos'] = y.sum()
    for feat in combine_features:
        auc_table.loc[f, feat] = roc_auc_score(y, df[feat].values)

    # combined_lr: 5-fold CV logistic regression
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)
    X = df[combine_features].values
    oof_pred = np.zeros(len(y))
    coefs = []
    for train_idx, test_idx in skf.split(X, y):
        scaler = StandardScaler().fit(X[train_idx])
        X_train = scaler.transform(X[train_idx])
        X_test = scaler.transform(X[test_idx])
        clf = LogisticRegression()
        clf.fit(X_train, y[train_idx])
        oof_pred[test_idx] = clf.predict_proba(X_test)[:, 1]
        coefs.append(clf.coef_[0])
    auc_table.loc[f, 'combined_lr'] = roc_auc_score(y, oof_pred)
    coef_table.loc[f] = np.mean(coefs, axis=0)

    # combined_rank: average percentile rank of the 3 features
    percentile_ranks = [df[feat].rank(pct=True).values for feat in combine_features]
    combined_rank = np.mean(percentile_ranks, axis=0)
    auc_table.loc[f, 'combined_rank'] = roc_auc_score(y, combined_rank)

print("=== AUC per feature, and combined via logistic regression / average percentile rank ===")
print(auc_table.T.round(4).to_string())
print()
print("=== avg standardized logistic regression coefficients (5-fold CV) ===")
print(coef_table.T.round(4).to_string())
